In [ ]:
from dotenv import load_dotenv
from anthropic import Anthropic
import json

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-0"

In [ ]:
# helper functions that will help maintain the conversation state with claude
def add_user_message(messages, text):
    user_message = {"role":"user","content":text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role":"assistant","content":text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    params={
        "model":model,
        "max_tokens":1000,
        "messages":messages,
        "temperature":temperature
    }
    if system:
        params["system"]=system
    if stop_sequences:
        params["stop_sequences"]=stop_sequences
        
    message=client.messages.create(**params)
    
    return message.content[0].text

messages=[]

In [ ]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
    Please solve the following task:
    {test_case["task"]}
    """
    add_user_message(messages,prompt)
    output = chat(messages)
    return output    

In [ ]:
import re,ast
#adding functions to validate json,python,regex generated
def validate_json(text):
    try:
        json.load(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0
    
def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0
    
def valiate_regex(text):
    try:
        re.parse(text.strip())
        return 10
    except SyntaxError:
        return 0
    
def grade_syntax(response, test_case):
    format = test_case["format"]
    if format=="json":
        return validate_json(response)
    elif format=="python":
        return validate_python(response)
    else:
        return valiate_regex(response)

In [ ]:
def grade_by_model(test_case, output):
    #we will make a call to the model and ask it to grade it.
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages.clear()
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [ ]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    #Grading done here
    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    return{
        "output":output,
        "test_case":test_case,
        "score":score,
        "reasoning":reasoning
    }

In [ ]:
from statistics import mean
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results=[]
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    #take the average of all the scores and then display
    average_score = mean([result["score"] for result in results])
    print(f"Average score is: {average_score}")
    return results

In [ ]:
with open("dataset.json","r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

print(json.dumps(results,indent=2))

In [ ]:
#just creating a file of results that came out of output
with open("results.json","w") as f:
    json.dump(results,f,indent=2)